# Model Pipeline Benchmark (GPU-Only)

Pipeline in this notebook:

1. (Optional) Training
2. Validation on the full test split
3. ONNX export (FP32)
4. ONNX export (FP16)
5. Speed benchmark on GPU
6. Final comparison table for `.pt`, `.onnx`, `.onnx (fp16)`

All benchmarks are GPU-only. The notebook requires `torch` CUDA and `onnxruntime-gpu` (`CUDAExecutionProvider`).



In [14]:
from pathlib import Path
import time
import shutil
from typing import Any

import cv2
import numpy as np
import yaml
import torch
import onnxruntime as ort
import pandas as pd

from ultralytics import YOLO



In [15]:
def detect_project_root() -> Path:
    candidates = [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]
    for candidate in candidates:
        if (candidate / "configs" / "train_config.yaml").exists():
            return candidate.resolve()
    raise FileNotFoundError(
        "Could not locate project root. Expected configs/train_config.yaml in cwd/parent folders. "
        f"Current cwd: {Path.cwd()}"
    )


PROJECT_ROOT = detect_project_root()
TRAIN_CONFIG_PATH = PROJECT_ROOT / "configs" / "train_config.yaml"

with TRAIN_CONFIG_PATH.open("r", encoding="utf-8") as f:
    cfg = yaml.safe_load(f) or {}

train_cfg = cfg.get("train", {})
eval_cfg = cfg.get("evaluate", {})
export_cfg = cfg.get("export", {})

RUN_TRAIN = False
RUN_EXPORT = True

BENCHMARK_DEVICE = "cuda"

weights_path = (PROJECT_ROOT / eval_cfg.get("weights", "models/weights/best.pt")).resolve()
if not weights_path.exists():
    latest_candidates = sorted(
        (PROJECT_ROOT / "runs" / "train").glob("*/weights/best.pt"),
        key=lambda p: p.stat().st_mtime,
        reverse=True,
    ) if (PROJECT_ROOT / "runs" / "train").exists() else []

    if latest_candidates:
        weights_path = latest_candidates[0]

if not weights_path.exists():
    raise FileNotFoundError(f"Weights not found: {weights_path}")


dataset_yaml = (PROJECT_ROOT / eval_cfg.get("data", train_cfg.get("data", "configs/dataset.yaml"))).resolve()
imgsz = int(eval_cfg.get("imgsz", train_cfg.get("imgsz", 640)))
batch = int(eval_cfg.get("batch", train_cfg.get("batch", 16)))
val_device = "0"  # ultralytics: GPU index
split = str(eval_cfg.get("split", "test"))

export_dir = (PROJECT_ROOT / export_cfg.get("output_dir", "models/onnx")).resolve()
export_dir.mkdir(parents=True, exist_ok=True)
onnx_path = export_dir / "model.onnx"
onnx_fp16_path = export_dir / "model.fp16.onnx"

reports_dir = (PROJECT_ROOT / "reports").resolve()
reports_dir.mkdir(parents=True, exist_ok=True)

with dataset_yaml.open("r", encoding="utf-8") as f:
    dataset_cfg_runtime = yaml.safe_load(f) or {}

dataset_root = Path(dataset_cfg_runtime.get("path", dataset_yaml.parent))
if not dataset_root.is_absolute():
    dataset_root = (PROJECT_ROOT / dataset_root).resolve()

dataset_cfg_runtime["path"] = str(dataset_root).replace("\\", "/")
dataset_yaml_runtime = reports_dir / "dataset_abs.yaml"
with dataset_yaml_runtime.open("w", encoding="utf-8") as f:
    yaml.safe_dump(dataset_cfg_runtime, f, allow_unicode=True, sort_keys=False)

available_ort_providers = ort.get_available_providers()
if not torch.cuda.is_available():
    raise RuntimeError("CUDA is not available for PyTorch. GPU-only benchmark cannot run.")
if "CUDAExecutionProvider" not in available_ort_providers:
    raise RuntimeError(
        "ONNX Runtime CUDAExecutionProvider is unavailable. Install onnxruntime-gpu to run GPU-only ONNX tests."
    )

print("project_root:", PROJECT_ROOT)
print("weights:", weights_path)
print("dataset_yaml_runtime:", dataset_yaml_runtime)
print("benchmark_device:", BENCHMARK_DEVICE)
print("val_device:", val_device)
print("split:", split)
print("ORT providers:", available_ort_providers)




project_root: C:\Users\sanek\OneDrive\Рабочий стол\cv_drone_pipeline
weights: C:\Users\sanek\OneDrive\Рабочий стол\cv_drone_pipeline\models\weights\best.pt
dataset_yaml_runtime: C:\Users\sanek\OneDrive\Рабочий стол\cv_drone_pipeline\reports\dataset_abs.yaml
benchmark_device: cuda
val_device: 0
split: test
ORT providers: ['TensorrtExecutionProvider', 'CUDAExecutionProvider', 'CPUExecutionProvider']


In [16]:
IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}


def read_unicode_image(path: Path):
    data = np.fromfile(str(path), dtype=np.uint8)
    if data.size == 0:
        return None
    return cv2.imdecode(data, cv2.IMREAD_COLOR)


def parse_val_metrics(results) -> dict[str, Any]:
    metrics = {}

    for k, v in (getattr(results, "results_dict", {}) or {}).items():
        if isinstance(v, (float, int)):
            metrics[str(k)] = float(v)

    box = getattr(results, "box", None)
    if box is not None:
        for attr, name in [
            ("mp", "precision"),
            ("mr", "recall"),
            ("map50", "map50"),
            ("map", "map50_95"),
        ]:
            value = getattr(box, attr, None)
            if value is not None:
                metrics[name] = float(value)

    return metrics


def model_size_mb(path: Path) -> float:
    if not path.exists():
        return float("nan")
    return path.stat().st_size / (1024 * 1024)


def rel_path(path: Path) -> str:
    try:
        return str(path.relative_to(PROJECT_ROOT)).replace("\\", "/")
    except Exception:
        return str(path).replace("\\", "/")


def get_split_images(dataset_yaml_path: Path, split_name: str = "test") -> list[Path]:
    with dataset_yaml_path.open("r", encoding="utf-8") as f:
        data_cfg = yaml.safe_load(f) or {}

    root = Path(data_cfg.get("path", dataset_yaml_path.parent))
    if not root.is_absolute():
        root = (PROJECT_ROOT / root).resolve()

    split_rel = data_cfg.get(split_name)
    if split_rel is None:
        raise ValueError(f"Split '{split_name}' not found in {dataset_yaml_path}")

    split_dir = (root / split_rel).resolve()
    if not split_dir.exists():
        raise FileNotFoundError(f"Split images dir not found: {split_dir}")

    images = sorted([p for p in split_dir.iterdir() if p.is_file() and p.suffix.lower() in IMAGE_EXTS])
    if not images:
        raise RuntimeError(f"No images found in {split_dir}")
    return images


def preprocess_for_onnx(image_path: Path, size: int) -> np.ndarray:
    image = read_unicode_image(image_path)
    if image is None:
        raise ValueError(f"Cannot read image: {image_path}")

    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    image = cv2.resize(image, (size, size), interpolation=cv2.INTER_LINEAR)
    image = image.astype(np.float32) / 255.0
    image = np.transpose(image, (2, 0, 1))[None, ...]
    return image


def benchmark_pt(weights: Path, tensors: list[np.ndarray], device_name: str = "cuda", warmup: int = 10, runs: int = 80):
    dev = torch.device(device_name if ("cuda" in device_name and torch.cuda.is_available()) else "cpu")
    model = YOLO(str(weights), task="detect").model.to(dev).eval()
    torch_tensors = [torch.from_numpy(t).to(dev) for t in tensors]

    with torch.no_grad():
        for i in range(warmup):
            _ = model(torch_tensors[i % len(torch_tensors)])
        if dev.type == "cuda":
            torch.cuda.synchronize()

        t0 = time.perf_counter()
        for i in range(runs):
            _ = model(torch_tensors[i % len(torch_tensors)])
        if dev.type == "cuda":
            torch.cuda.synchronize()
        t1 = time.perf_counter()

    latency_ms = (t1 - t0) * 1000.0 / runs
    return {
        "runtime": f"torch:{dev.type}",
        "latency_ms": latency_ms,
        "fps": 1000.0 / latency_ms if latency_ms > 0 else float("nan"),
    }


def _onnx_expected_dtype(input_type: str):
    t = (input_type or "").lower()
    if "float16" in t:
        return np.float16
    if "double" in t:
        return np.float64
    return np.float32


def benchmark_onnx(model_path: Path, tensors: list[np.ndarray], warmup: int = 10, runs: int = 80):
    sess = ort.InferenceSession(str(model_path), providers=["CUDAExecutionProvider"])
    providers = sess.get_providers()
    if not providers or providers[0] != "CUDAExecutionProvider":
        raise RuntimeError(f"ONNX session is not running on CUDA. Providers: {providers}")

    input_meta = sess.get_inputs()[0]
    input_name = input_meta.name
    expected_dtype = _onnx_expected_dtype(getattr(input_meta, "type", "tensor(float)"))

    cast_tensors = [np.asarray(t, dtype=expected_dtype) for t in tensors]

    for i in range(warmup):
        _ = sess.run(None, {input_name: cast_tensors[i % len(cast_tensors)]})

    t0 = time.perf_counter()
    for i in range(runs):
        _ = sess.run(None, {input_name: cast_tensors[i % len(cast_tensors)]})
    t1 = time.perf_counter()

    latency_ms = (t1 - t0) * 1000.0 / runs
    return {
        "runtime": "onnxruntime:" + ",".join(providers),
        "input_dtype": str(expected_dtype).replace("<class '", "").replace("'>", ""),
        "latency_ms": latency_ms,
        "fps": 1000.0 / latency_ms if latency_ms > 0 else float("nan"),
    }




In [17]:
dataset_yaml_eval = dataset_yaml_runtime
sample_images = get_split_images(dataset_yaml_eval, split_name=split)

input_tensors = [preprocess_for_onnx(p, imgsz) for p in sample_images]
print(f"Prepared {len(input_tensors)} tensors for benchmark/val (full split='{split}')")
print("dataset_yaml_eval:", dataset_yaml_eval)



Prepared 579 tensors for benchmark/val (full split='test')
dataset_yaml_eval: C:\Users\sanek\OneDrive\Рабочий стол\cv_drone_pipeline\reports\dataset_abs.yaml


In [18]:
if RUN_TRAIN:
    train_args = train_cfg.copy()
    model_name = train_args.pop("model", "yolo11n.pt")
    trainer = YOLO(model_name)
    trainer.train(**train_args)
    print("Training finished")
else:
    print("Training skipped")


Training skipped


In [19]:
pt_model = YOLO(str(weights_path), task="detect")
pt_val = pt_model.val(
    data=str(dataset_yaml_eval),
    split=split,
    imgsz=imgsz,
    batch=batch,
    device=val_device,
    verbose=False,
)
pt_metrics = parse_val_metrics(pt_val)
pt_metrics



Ultralytics 8.3.145  Python-3.12.3 torch-2.7.1+cu128 CUDA:0 (NVIDIA GeForce GTX 1080, 8192MiB)
YOLO11n summary (fused): 100 layers, 2,582,347 parameters, 0 gradients, 6.3 GFLOPs
val: Fast image access  (ping: 0.00.0 ms, read: 584.7414.7 MB/s, size: 60.3 KB)


val: Scanning C:\Users\sanek\OneDrive\Рабочий стол\cv_drone_pipeline\data\processed\YOLO\labels\test.cache... 579 images, 92 backgrounds, 0 corrupt: 100%|██████████| 579/579 [00:00<?, ?it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 37/37 [00:04<00:00,  9.17it/s]


                   all        579        845      0.513      0.347      0.356      0.166
Speed: 0.2ms preprocess, 2.7ms inference, 0.0ms loss, 1.3ms postprocess per image
Results saved to runs\detect\val34


{'metrics/precision(B)': 0.512575668682112,
 'metrics/recall(B)': 0.3467455621301775,
 'metrics/mAP50(B)': 0.3557961578292047,
 'metrics/mAP50-95(B)': 0.16581011196793313,
 'fitness': 0.18480871655406028,
 'precision': 0.512575668682112,
 'recall': 0.3467455621301775,
 'map50': 0.3557961578292047,
 'map50_95': 0.16581011196793313}

In [20]:
# Export baseline ONNX
if RUN_EXPORT or not onnx_path.exists():
    exported = pt_model.export(
        format="onnx",
        imgsz=int(export_cfg.get("imgsz", imgsz)),
        dynamic=bool(export_cfg.get("dynamic", False)),
        simplify=bool(export_cfg.get("simplify", True)),
        half=False,
        device=0,
    )
    exported_path = Path(str(exported)).resolve()
    if exported_path != onnx_path:
        shutil.copy2(exported_path, onnx_path)

# Export FP16 ONNX (GPU-only)
if RUN_EXPORT or not onnx_fp16_path.exists():
    exported_fp16 = pt_model.export(
        format="onnx",
        imgsz=int(export_cfg.get("imgsz", imgsz)),
        dynamic=bool(export_cfg.get("dynamic", False)),
        simplify=bool(export_cfg.get("simplify", True)),
        half=True,
        device=0,
    )
    exported_fp16_path = Path(str(exported_fp16)).resolve()
    if exported_fp16_path != onnx_fp16_path:
        shutil.copy2(exported_fp16_path, onnx_fp16_path)

sess_fp32 = ort.InferenceSession(str(onnx_path), providers=["CUDAExecutionProvider"])
sess_fp16 = ort.InferenceSession(str(onnx_fp16_path), providers=["CUDAExecutionProvider"])
print("ONNX:", onnx_path, "exists:", onnx_path.exists(), "size_mb:", round(model_size_mb(onnx_path), 2), "input:", sess_fp32.get_inputs()[0].type)
print("ONNX FP16:", onnx_fp16_path, "exists:", onnx_fp16_path.exists(), "size_mb:", round(model_size_mb(onnx_fp16_path), 2), "input:", sess_fp16.get_inputs()[0].type)



Ultralytics 8.3.145  Python-3.12.3 torch-2.7.1+cu128 CUDA:0 (NVIDIA GeForce GTX 1080, 8192MiB)

PyTorch: starting from 'C:\Users\sanek\OneDrive\ \cv_drone_pipeline\models\weights\best.pt' with input shape (1, 3, 640, 640) BCHW and output shape(s) (1, 5, 8400) (5.2 MB)

ONNX: starting export with onnx 1.17.0 opset 19...
ONNX: slimming with onnxslim 0.1.58...
ONNX: export success  0.9s, saved as 'C:\Users\sanek\OneDrive\ \cv_drone_pipeline\models\weights\best.onnx' (10.1 MB)

Export complete (1.2s)
Results saved to C:\Users\sanek\OneDrive\ \cv_drone_pipeline\models\weights
Predict:         yolo predict task=detect model=C:\Users\sanek\OneDrive\ \cv_drone_pipeline\models\weights\best.onnx imgsz=640  
Validate:        yolo val task=detect model=C:\Users\sanek\OneDrive\ \cv_drone_pipeline\models\weights\best.onnx imgsz=640 data=data/train_data/data.yaml  
Visualize:       https://netron.app
Ultralytics 8.3.145  Python-3.12.3 torch-2.7.1+cu128 CUDA:0 (NVIDIA GeForce GTX 1080, 8192MiB)

PyTor

In [21]:
onnx_metrics = {}
onnx_fp16_metrics = {}

try:
    onnx_model = YOLO(str(onnx_path), task="detect")
    onnx_val = onnx_model.val(
        data=str(dataset_yaml_eval),
        split=split,
        imgsz=imgsz,
        batch=batch,
        device=0,
        verbose=False,
    )
    onnx_metrics = parse_val_metrics(onnx_val)
except Exception as exc:
    print("ONNX validation failed:", repr(exc))

try:
    onnx_fp16_model = YOLO(str(onnx_fp16_path), task="detect")
    onnx_fp16_val = onnx_fp16_model.val(
        data=str(dataset_yaml_eval),
        split=split,
        imgsz=imgsz,
        batch=batch,
        device=0,
        verbose=False,
    )
    onnx_fp16_metrics = parse_val_metrics(onnx_fp16_val)
except Exception as exc:
    print("ONNX FP16 validation failed:", repr(exc))

print("onnx_metrics:", onnx_metrics)
print("onnx_fp16_metrics:", onnx_fp16_metrics)



Ultralytics 8.3.145  Python-3.12.3 torch-2.7.1+cu128 CUDA:0 (NVIDIA GeForce GTX 1080, 8192MiB)
Loading C:\Users\sanek\OneDrive\ \cv_drone_pipeline\models\onnx\model.onnx for ONNX Runtime inference...
Using ONNX Runtime CUDAExecutionProvider
Setting batch=1 input of shape (1, 3, 640, 640)
val: Fast image access  (ping: 0.00.0 ms, read: 777.7143.0 MB/s, size: 92.8 KB)


val: Scanning C:\Users\sanek\OneDrive\Рабочий стол\cv_drone_pipeline\data\processed\YOLO\labels\test.cache... 579 images, 92 backgrounds, 0 corrupt: 100%|██████████| 579/579 [00:00<?, ?it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 579/579 [00:07<00:00, 75.48it/s]


                   all        579        845      0.494      0.349      0.354      0.165
Speed: 0.4ms preprocess, 8.5ms inference, 0.0ms loss, 1.6ms postprocess per image
Results saved to runs\detect\val35
Ultralytics 8.3.145  Python-3.12.3 torch-2.7.1+cu128 CUDA:0 (NVIDIA GeForce GTX 1080, 8192MiB)
Loading C:\Users\sanek\OneDrive\ \cv_drone_pipeline\models\onnx\model.fp16.onnx for ONNX Runtime inference...
Using ONNX Runtime CUDAExecutionProvider
Setting batch=1 input of shape (1, 3, 640, 640)
val: Fast image access  (ping: 0.00.0 ms, read: 708.4229.8 MB/s, size: 69.5 KB)


val: Scanning C:\Users\sanek\OneDrive\Рабочий стол\cv_drone_pipeline\data\processed\YOLO\labels\test.cache... 579 images, 92 backgrounds, 0 corrupt: 100%|██████████| 579/579 [00:00<?, ?it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 579/579 [00:09<00:00, 60.70it/s]


                   all        579        845      0.494      0.353      0.356      0.166
Speed: 0.4ms preprocess, 11.2ms inference, 0.0ms loss, 1.8ms postprocess per image
Results saved to runs\detect\val36
onnx_metrics: {'metrics/precision(B)': 0.4940473274093936, 'metrics/recall(B)': 0.34911242603550297, 'metrics/mAP50(B)': 0.35404841608608706, 'metrics/mAP50-95(B)': 0.1648364954979808, 'fitness': 0.18375768755679142, 'precision': 0.4940473274093936, 'recall': 0.34911242603550297, 'map50': 0.35404841608608706, 'map50_95': 0.1648364954979808}
onnx_fp16_metrics: {'metrics/precision(B)': 0.4943899975008231, 'metrics/recall(B)': 0.3529373160142391, 'metrics/mAP50(B)': 0.3562938226983525, 'metrics/mAP50-95(B)': 0.16561640089194518, 'fitness': 0.18468414307258593, 'precision': 0.4943899975008231, 'recall': 0.3529373160142391, 'map50': 0.3562938226983525, 'map50_95': 0.16561640089194518}


In [22]:
rows = []

pt_bench = benchmark_pt(weights_path, input_tensors, device_name=BENCHMARK_DEVICE, warmup=10, runs=80)
rows.append(
    {
        "model_format": ".pt",
        "path": rel_path(weights_path),
        "size_mb": round(model_size_mb(weights_path), 2),
        "precision": round(float(pt_metrics.get("precision", float("nan"))), 4),
        "recall": round(float(pt_metrics.get("recall", float("nan"))), 4),
        "map50": round(float(pt_metrics.get("map50", float("nan"))), 4),
        "map50_95": round(float(pt_metrics.get("map50_95", float("nan"))), 4),
        "latency_ms": round(pt_bench["latency_ms"], 3),
        "fps": round(pt_bench["fps"], 2),
        "runtime": pt_bench["runtime"],
        "input_dtype": "float32",
    }
)

onnx_bench = benchmark_onnx(onnx_path, input_tensors, warmup=10, runs=80)
rows.append(
    {
        "model_format": ".onnx",
        "path": rel_path(onnx_path),
        "size_mb": round(model_size_mb(onnx_path), 2),
        "precision": round(float(onnx_metrics.get("precision", float("nan"))), 4),
        "recall": round(float(onnx_metrics.get("recall", float("nan"))), 4),
        "map50": round(float(onnx_metrics.get("map50", float("nan"))), 4),
        "map50_95": round(float(onnx_metrics.get("map50_95", float("nan"))), 4),
        "latency_ms": round(onnx_bench["latency_ms"], 3),
        "fps": round(onnx_bench["fps"], 2),
        "runtime": onnx_bench["runtime"],
        "input_dtype": onnx_bench.get("input_dtype", "n/a"),
    }
)

onnx_fp16_bench = benchmark_onnx(onnx_fp16_path, input_tensors, warmup=10, runs=80)
rows.append(
    {
        "model_format": ".onnx (fp16)",
        "path": rel_path(onnx_fp16_path),
        "size_mb": round(model_size_mb(onnx_fp16_path), 2),
        "precision": round(float(onnx_fp16_metrics.get("precision", float("nan"))), 4),
        "recall": round(float(onnx_fp16_metrics.get("recall", float("nan"))), 4),
        "map50": round(float(onnx_fp16_metrics.get("map50", float("nan"))), 4),
        "map50_95": round(float(onnx_fp16_metrics.get("map50_95", float("nan"))), 4),
        "latency_ms": round(onnx_fp16_bench["latency_ms"], 3),
        "fps": round(onnx_fp16_bench["fps"], 2),
        "runtime": onnx_fp16_bench["runtime"],
        "input_dtype": onnx_fp16_bench.get("input_dtype", "n/a"),
    }
)

comparison_gpu_df = pd.DataFrame(rows)
comparison_gpu_df



,model_format,path,size_mb,precision,recall,map50,map50_95,latency_ms,fps,runtime,input_dtype
0,.pt,models/weights/best.pt,5.19,0.5126,0.3467,0.3558,0.1658,12.701,78.74,torch:cuda,float32
1,.onnx,models/onnx/model.onnx,10.09,0.4940,0.3491,0.3540,0.1648,9.094,109.97,"onnxruntime:CUDAExecutionProvider,CPUExecution...",numpy.float32
2,.onnx (fp16),models/onnx/model.fp16.onnx,5.09,0.4944,0.3529,0.3563,0.1656,11.965,83.57,"onnxruntime:CUDAExecutionProvider,CPUExecution...",numpy.float16


In [23]:
comparison_gpu_df.sort_values("latency_ms")


,model_format,path,size_mb,precision,recall,map50,map50_95,latency_ms,fps,runtime,input_dtype
1,.onnx,models/onnx/model.onnx,10.09,0.4940,0.3491,0.3540,0.1648,9.094,109.97,"onnxruntime:CUDAExecutionProvider,CPUExecution...",numpy.float32
2,.onnx (fp16),models/onnx/model.fp16.onnx,5.09,0.4944,0.3529,0.3563,0.1656,11.965,83.57,"onnxruntime:CUDAExecutionProvider,CPUExecution...",numpy.float16
0,.pt,models/weights/best.pt,5.19,0.5126,0.3467,0.3558,0.1658,12.701,78.74,torch:cuda,float32


In [24]:
out_gpu_csv = reports_dir / "model_comparison_gpu.csv"
out_gpu_md = reports_dir / "model_comparison_gpu.md"

comparison_gpu_df.to_csv(out_gpu_csv, index=False)
with out_gpu_md.open("w", encoding="utf-8") as f:
    f.write(comparison_gpu_df.to_markdown(index=False))

print("Saved GPU comparison:")
print("-", out_gpu_csv)
print("-", out_gpu_md)

comparison_gpu_df.sort_values("latency_ms")



Saved GPU comparison:
- C:\Users\sanek\OneDrive\Рабочий стол\cv_drone_pipeline\reports\model_comparison_gpu.csv
- C:\Users\sanek\OneDrive\Рабочий стол\cv_drone_pipeline\reports\model_comparison_gpu.md


,model_format,path,size_mb,precision,recall,map50,map50_95,latency_ms,fps,runtime,input_dtype
1,.onnx,models/onnx/model.onnx,10.09,0.4940,0.3491,0.3540,0.1648,9.094,109.97,"onnxruntime:CUDAExecutionProvider,CPUExecution...",numpy.float32
2,.onnx (fp16),models/onnx/model.fp16.onnx,5.09,0.4944,0.3529,0.3563,0.1656,11.965,83.57,"onnxruntime:CUDAExecutionProvider,CPUExecution...",numpy.float16
0,.pt,models/weights/best.pt,5.19,0.5126,0.3467,0.3558,0.1658,12.701,78.74,torch:cuda,float32


## Notes

- This version intentionally removes INT8 quantization to keep the comparison focused and stable.
- Validation and speed benchmark use the full split from `dataset.yaml` (no subset limit).
- ONNX sessions are created with `providers=["CUDAExecutionProvider"]`; execution stops if CUDA provider is unavailable.



In [25]:
summary_rows = []

base = comparison_gpu_df.loc[comparison_gpu_df["model_format"] == ".pt"].iloc[0]
for _, row in comparison_gpu_df.iterrows():
    speedup_vs_pt = row["fps"] / base["fps"] if base["fps"] > 0 else float("nan")
    delta_map50 = row["map50"] - base["map50"]
    delta_map5095 = row["map50_95"] - base["map50_95"]
    summary_rows.append(
        {
            "model_format": row["model_format"],
            "fps": round(float(row["fps"]), 2),
            "speedup_vs_pt_x": round(float(speedup_vs_pt), 3),
            "map50": round(float(row["map50"]), 4),
            "map50_95": round(float(row["map50_95"]), 4),
            "delta_map50_vs_pt": round(float(delta_map50), 4),
            "delta_map50_95_vs_pt": round(float(delta_map5095), 4),
        }
    )

summary_df = pd.DataFrame(summary_rows).sort_values("fps", ascending=False)
summary_df



,model_format,fps,speedup_vs_pt_x,map50,map50_95,delta_map50_vs_pt,delta_map50_95_vs_pt
1,.onnx,109.97,1.397,0.3540,0.1648,-0.0018,-0.0010
2,.onnx (fp16),83.57,1.061,0.3563,0.1656,0.0005,-0.0002
0,.pt,78.74,1.000,0.3558,0.1658,0.0000,0.0000


In [26]:
pt_row = comparison_gpu_df.loc[comparison_gpu_df["model_format"] == ".pt"].iloc[0]
onnx_row = comparison_gpu_df.loc[comparison_gpu_df["model_format"] == ".onnx"].iloc[0]
onnx_fp16_row = comparison_gpu_df.loc[comparison_gpu_df["model_format"] == ".onnx (fp16)"].iloc[0]

fastest_row = comparison_gpu_df.sort_values("fps", ascending=False).iloc[0]

print("Fastest model:", fastest_row["model_format"])
print(f"ONNX FP32 speedup vs PT: {onnx_row['fps']/pt_row['fps']:.2f}x")
print(f"ONNX FP16 speedup vs PT: {onnx_fp16_row['fps']/pt_row['fps']:.2f}x")
print(f"mAP50 delta ONNX FP32 vs PT: {onnx_row['map50'] - pt_row['map50']:+.4f}")
print(f"mAP50-95 delta ONNX FP32 vs PT: {onnx_row['map50_95'] - pt_row['map50_95']:+.4f}")
print(f"mAP50 delta ONNX FP16 vs PT: {onnx_fp16_row['map50'] - pt_row['map50']:+.4f}")
print(f"mAP50-95 delta ONNX FP16 vs PT: {onnx_fp16_row['map50_95'] - pt_row['map50_95']:+.4f}")



Fastest model: .onnx
ONNX FP32 speedup vs PT: 1.40x
ONNX FP16 speedup vs PT: 1.06x
mAP50 delta ONNX FP32 vs PT: -0.0018
mAP50-95 delta ONNX FP32 vs PT: -0.0010
mAP50 delta ONNX FP16 vs PT: +0.0005
mAP50-95 delta ONNX FP16 vs PT: -0.0002


## Final Conclusions

### Key Findings
- On this hardware, **`.onnx` (FP32)** is the most practical model for deployment.
- It is significantly faster than **`.pt`** while keeping quality very close.
- **`.onnx (fp16)`** did not provide a speed gain over FP32 ONNX in this setup.

### Why ONNX is faster than PT
- ONNX Runtime executes an optimized static graph and reduces Python/PyTorch eager overhead.
- For inference workloads, this often gives lower latency and higher FPS at similar accuracy.

### Why ONNX FP16 can be slower than ONNX FP32
- My GPU (**GTX 1080, Pascal**) has no Tensor Cores, so FP16 is not guaranteed to be faster.
- Some operators may run with less efficient kernels or require extra type conversions.
- As a result, FP16 benefits are hardware-dependent and must be validated empirically.

### Speed vs Quality Trade-off
Relative to **`.pt`**:
- **`.onnx` (FP32)**: strong FPS gain with only a small drop in mAP.
- **`.onnx (fp16)`**: lower speed than FP32 ONNX, with no clear quality advantage.

### Production Recommendation (Edge Inference)
- Use **`.onnx` (FP32)** as the primary production model on this device.
- Keep **`.pt`** as a quality baseline and fallback option when maximum accuracy is more important than latency.
- Re-test FP16 only on GPUs with Tensor Cores (e.g., RTX/T4/A10/L4), where FP16 usually performs better.

___

## Фiнальний результат

### Ключові висновки
- На цьому залізі **`.onnx` (FP32)** є найкращим практичним варіантом для деплою.
- Він суттєво швидший за **`.pt`**, при цьому якість залишається дуже близькою.
- **`.onnx (fp16)`** у цьому сетапі не дав приросту швидкості відносно FP32 ONNX.

### Чому ONNX швидший за PT
- ONNX Runtime виконує оптимізований статичний граф і зменшує накладні витрати Python/PyTorch eager.
- Для inference-навантажень це часто дає меншу затримку та вищий FPS при схожій точності.

### Чому ONNX FP16 може бути повільнішим за ONNX FP32
- Моя GPU (**GTX 1080, Pascal**) не має Tensor Cores, тому FP16 не гарантує прискорення.
- Частина операторів може працювати на менш ефективних kernel’ах або вимагати додаткових перетворень типів.
- Тому виграш від FP16 залежить від заліза і завжди має перевірятися бенчмарком.

### Компроміс швидкість/якість
Відносно **`.pt`**:
- **`.onnx` (FP32)**: помітний приріст FPS при невеликій втраті mAP.
- **`.onnx (fp16)`**: нижча швидкість, ніж у FP32 ONNX, без явної переваги в якості.

### Рекомендація для production (edge inference)
- Використовуй **`.onnx` (FP32)** як основну production-модель на цьому пристрої.
- **`.pt`** залиш як quality baseline та резервний варіант, якщо пріоритет зміщується в бік максимальної якості.
- FP16 варто повторно тестувати на GPU з Tensor Cores (наприклад, RTX/T4/A10/L4), де він зазвичай показує кращий результат.